In [1]:
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from navier_stokes_common import PINNsformer, init_weights, get_n_params, load_training_data, compute_ns_losses, evaluation_tensors

# 和之前保持一致
SEED = 0
DEVICE = "cuda:0"
N_TRAIN = 800
NUM_STEP = 5
TIME_STEP = 1e-2

device = torch.device(DEVICE)

PINNSFORMER_ROOT = Path("/home/simplexity/cyt/pinnsformer-main")
DATA_PATH = (
    PINNSFORMER_ROOT
    / "demo"
    / "navier_stokes"
    / "cylinder_nektar_wake.mat"
)

MODEL_PATH = Path(
    "./outputs/navier_stokes_config/"
    "ns_pinnsformer_config_2loss.pt"
)

In [2]:
model = PINNsformer(
    d_out=2,
    d_hidden=512,
    d_model=32,
    N=1,
    heads=2
).to(device)

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

model.eval()
print("Loaded:", MODEL_PATH)

NameError: name 'PINNsformer' is not defined

In [ ]:
psi_p = model(
    ev["x"],
    ev["y"],
    ev["t"]
)

psi = psi_p[:, :, 0:1]
p_pred = psi_p[:, :, 1:2]

# 注意：保持与训练定义一致
# u = psi_y
# v = -psi_x
u_pred = torch.autograd.grad(
    psi,
    ev["y"],
    torch.ones_like(psi),
    retain_graph=True,
)[0]

v_pred = -torch.autograd.grad(
    psi,
    ev["x"],
    torch.ones_like(psi),
)[0]

u_pred = u_pred.detach().cpu().numpy()[:, 0]
v_pred = v_pred.detach().cpu().numpy()[:, 0]
p_pred = p_pred.detach().cpu().numpy()[:, 0]

In [ ]:
preds = {
    "u": u_pred,
    "v": v_pred,
    "p": p_pred,
}

for name in ["u", "v", "p"]:
    error = np.abs(
        truth[name].reshape(-1)
        - preds[name].reshape(-1)
    )

    plt.figure(figsize=(5, 3.5))
    plt.imshow(
        error.reshape(50, 100),
        extent=[-3, 8, -2, 2],
        aspect="auto",
        origin="lower",
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{name} Absolute Error - PINNsFormer + ConFIG")
    plt.colorbar(label="Absolute Error")
    plt.tight_layout()

    plt.savefig(
        f"config_{name}_absolute_error.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()